In [1]:
from math import sqrt
import transformers
from transformers import AutoModel, AutoTokenizer, AutoConfig
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)
config = AutoConfig.from_pretrained(model_ckpt)
text = "time flies like an arrow"
print(model)

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [3]:
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
inputs.input_ids

tensor([[ 2051, 10029,  2066,  2019,  8612]])

In [4]:
x = sum(p.numel() for p in model.parameters())

In [5]:
x

66362880

In [6]:
def scaled_dot_product_attention(query, key, value):
    dim_k = value.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
    weights = F.softmax(scores, dim=-1)
    attn = torch.bmm(weights, value)
    return attn

In [7]:
class AttentionHead(nn.Module):
    def __init__(self, emb_dim, head_dim):
        super().__init__()
        self.q = nn.Linear(emb_dim, head_dim)
        self.k = nn.Linear(emb_dim, head_dim)
        self.v = nn.Linear(emb_dim, head_dim)

    def forward(self, hidden_state):
        attn_outputs = scaled_dot_product_attention(
            self.q(hidden_state), self.k(hidden_state), self.v(hidden_state)
        )
        return attn_outputs

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        emb_dim = config.dim
        num_head = config.n_heads
        head_dim = emb_dim // num_head
        self.heads = nn.ModuleList(
            [AttentionHead(emb_dim, head_dim) for _ in range(num_head)]
        )
        self.linear_output = nn.Linear(head_dim*num_head, emb_dim) 
    def forward(self, hidden_state):
        x = torch.cat([h(hidden_state) for h in self.heads], dim=-1)
        x = self.linear_output(x)
        return x

In [9]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.linear1 = nn.Linear(config.dim, config.hidden_dim)
        self.linear2 = nn.Linear(config.hidden_dim, config.dim)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.linear1(x)
        x = self.gelu(x)
        x = self.linear2(x)
        x = self.dropout(x)
        return x

In [10]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(config.dim)
        self.layer_norm2 = nn.LayerNorm(config.dim)
        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)
    def forward(self, x):
        hidden_state = self.layer_norm1(x)
        x = x + self.attention(hidden_state)
        x = x + self.feed_forward(self.layer_norm2(x))
        return x

In [11]:
class Embeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(
            config.vocab_size, config.dim
        )
        self.positional_embedding = nn.Embedding(
            config.max_position_embeddings, config.dim
        )
        self.layer_norm = nn.LayerNorm(config.dim, eps=1e-12)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        seq_length = input_ids.size(1)
        postion_ids = torch.arange(seq_length, dtype=torch.long).unsqueeze(0)
        token_embedding = self.token_embedding(input_ids)
        positional_embedding = self.positional_embedding(postion_ids)
        embeddings = token_embedding + positional_embedding
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

In [12]:
embedding = Embeddings(config)
embedding(inputs.input_ids).size()

torch.Size([1, 5, 768])

In [13]:
embedding

Embeddings(
  (token_embedding): Embedding(30522, 768)
  (positional_embedding): Embedding(512, 768)
  (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [14]:
class TransformerEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embedding = Embeddings(config)
        self.layers = nn.ModuleList(
            [TransformerEncoderLayer(config) for _ in range(config.n_layers)]
        )

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)

        return x

In [15]:
transformer = TransformerEncoder(config)
outputs = transformer(inputs.input_ids)

In [1]:
import mlflow

In [2]:
mlflow.set_tracking_uri("http:localhost:5000")